# PyTorch Dataloading with NanoTST — Part 2: torchdata.nodes

**Today we're rebuilding Part 1's pipeline with `torchdata.nodes`** — PyTorch's unbundling of the DataLoader into composable iterator nodes.

By the end you'll understand:

1. **Why the DataLoader is being taken apart** — the four pains of the opaque box
2. **The map-style port** — our DataLoader from Part 1, rebuilt line-for-line as nodes
3. **The killer feature** — `state_dict()` on the whole pipeline: mid-epoch checkpoint and exact resume
4. **The iterable port** — our infinite stream as a custom node, where the sharding problem *dissolves* — and even an infinite stream becomes resumable

Recall Part 1's map — nodes explode the middle box:

```
Dataset ──(sampler)──> [ DataLoader: workers + collate ] ──> batch ──> model
                        └── becomes: Sampler → Batcher → ParallelMapper → Prefetcher ──┘
```

## Why unbundle? Four pains of the opaque box

1. **Multiprocessing duplicates memory** — each worker process clones your dataset (Python copy-on-read), and batches pay IPC costs coming back
2. **Samplers break down with multiple datasets** — no clean weighted mixing / round-robin
3. **IterableDataset makes you shard by hand** — Part 1's `get_worker_info()` dance
4. **No pipeline state** — mid-epoch resume means hacky skip-counters

The redesign: **stop cloning the reader.** One iterator runs in the main process; parallelism is applied *inside* the map step, to work items. Every node is an iterator with three methods — `next()`, `get_state()`, `reset(state)`. Generators are banned on purpose: state must be explicit.

In [1]:
import torch
import torchdata
import torchdata.nodes as tn
from torch.utils.data import RandomSampler, default_collate

import tst_data
from nano_tst import NanoTST
from tst_data import (SeriesDataset, SeriesStreamNode, EpochSeededRandomSampler,
                      train_with_loader)

torch.manual_seed(42)
print("torch", torch.__version__, "| torchdata", torchdata.__version__)

torch 2.2.2 | torchdata 0.11.0+cpu


## The map-style port — same pipeline, named boxes

In Part 1 this was `DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True, num_workers=N)`.
Here is the same machine with the lid off — **each line is one box in the diagram**:

In [2]:
class MapAndCollate:
    """Given a batch of indices: fetch each sample from the dataset, collate into one tensor.
    (This is the work DataLoader workers were doing behind the curtain.)"""
    def __init__(self, dataset, collate_fn=default_collate):
        self.dataset = dataset
        self.collate_fn = collate_fn
    def __call__(self, indices):
        return self.collate_fn([self.dataset[i] for i in indices])


def nodes_dataloader(dataset, batch_size, shuffle=True, num_workers=2, drop_last=True):
    # Not RandomSampler: exact resume needs *reconstructible* randomness.
    # RandomSampler draws a fresh permutation every iteration, so a restored
    # pipeline would fast-forward through the wrong order. Seeding by epoch
    # makes the permutation replayable — same moral as banning generators.
    sampler = EpochSeededRandomSampler(dataset)            # shuffle=True
    node = tn.SamplerWrapper(sampler)                      # indices, one at a time
    node = tn.Batcher(node, batch_size=batch_size, drop_last=drop_last)   # lists of indices
    node = tn.ParallelMapper(                              # THE parallelism — on work items,
        node,                                              # not on cloned readers
        map_fn=MapAndCollate(dataset),
        num_workers=num_workers,
        method="thread",                                   # or "process" — your choice now
        in_order=True,
    )
    node = tn.Prefetcher(node, prefetch_factor=2)          # overlap loading with training
    return tn.Loader(node)                                 # a normal iterable, auto-reset per epoch


dataset = SeriesDataset(n_series=512)
loader = nodes_dataloader(dataset, batch_size=32)
batch = next(iter(loader))
print(f"batch.shape = {batch.shape}")   # same [32, 512] as Part 1

batch.shape = torch.Size([32, 512])


Notes on that cell:

- **`method="thread"`** is the quiet headline. Workers can be threads: no dataset cloning, no IPC. Works today when the per-sample work is C-library territory (tensor ops, decode); becomes fully general with free-threaded Python 3.13+.
- **`PinMemory`** would slot in after the mapper on a GPU machine (`node = tn.PinMemory(node)`); skipped here on CPU.
- The parallelism knob moved from "how many copies of everything" (`num_workers`) to "how many hands on the map step".


In [3]:
# Same model, same training loop as Part 1 — only the loader construction changed
model = NanoTST()
losses = train_with_loader(model, loader, epochs=3)

  Epoch 1 | loss: 1.6323 (16 steps)


  Epoch 2 | loss: 1.3782 (16 steps)


  Epoch 3 | loss: 1.2399 (16 steps)


## The killer feature: checkpoint the pipeline mid-epoch

`loader.state_dict()` snapshots the position of *every node* — sampler order included. The DataLoader never had this; everyone hacked it with skip-counters.

One catch that teaches the design: on restore, `SamplerWrapper` re-iterates your sampler and fast-forwards. A plain `RandomSampler` draws a *fresh* permutation each iteration — the pipeline would resume into the wrong order. That's why `nodes_dataloader` uses the epoch-seeded sampler: **exact resume requires reconstructible randomness** — the same reason nodes ban generators and force `get_state()` to be explicit.

In [4]:
loader = nodes_dataloader(dataset, batch_size=32)

seen, resumed = [], []
for step, batch in enumerate(loader):
    if step == 3:
        checkpoint = loader.state_dict()     # <- snapshot mid-epoch: batches 0-3 delivered
    elif step > 3:
        seen.append(batch)                   # what the rest of the epoch looked like
    if step == 6:
        break

# "Crash" and come back: restore, and the pipeline continues at batch 4 exactly
loader.load_state_dict(checkpoint)
for step, batch in enumerate(loader):
    resumed.append(batch)
    if step == 2:
        break

print("resumed exactly where we left off:",
      all(torch.equal(a, b) for a, b in zip(seen, resumed)))

resumed exactly where we left off: True


## The iterable port — the sharding problem dissolves

Part 1's infinite stream needed `get_worker_info()` gymnastics because **every worker owned a full copy of the iterator**. In nodes, no worker ever owns the stream: the source iterates once, in the main process, and workers are only handed items to process. There's nothing to shard.

torchdata has no `IterableDataset` wrapper yet (officially "coming soon") — but we don't need one. We port the stream as a **custom node**: subclass `BaseNode`, implement `next()` / `get_state()` / `reset()`. Our stream's entire state is one integer:

In [5]:
import inspect
print(inspect.getsource(SeriesStreamNode))

    class SeriesStreamNode(BaseNode[torch.Tensor]):
        """The infinite stream, rebuilt as a node.

        No generators, no hidden position: the entire state is one integer.
        That's what makes an *infinite* stream checkpointable — get_state()
        returns {"index": i}, reset(state) puts you back on sample i exactly.
        """

        def __init__(self, seed: int = 77, length: int = CONTEXT_LEN):
            super().__init__()
            self.seed = seed
            self.length = length
            self.index = 0

        def reset(self, initial_state: dict | None = None):
            super().reset(initial_state)
            self.index = initial_state["index"] if initial_state else 0

        def next(self) -> torch.Tensor:
            sample = series_at(self.seed, self.index, self.length)
            self.index += 1
            return sample

        def get_state(self) -> dict:
            return {"index": self.index}



In [6]:
# The infinite stream, as a pipeline: source -> parallel per-sample work -> batch -> stack
node = SeriesStreamNode(seed=77)
node = tn.ParallelMapper(node, map_fn=lambda s: (s - s.mean()) / s.std(),   # any per-sample work,
                         num_workers=2, method="thread")                    # parallel, unsharded
node = tn.Batcher(node, batch_size=32)
node = tn.Mapper(node, map_fn=torch.stack)     # collate
stream_loader = tn.Loader(node)

batch = next(iter(stream_loader))
print(f"stream batch: {batch.shape} — two workers, zero duplication, zero sharding code")

stream batch: torch.Size([32, 512]) — two workers, zero duplication, zero sharding code


In [7]:
# And because state is explicit, even the INFINITE stream is resumable:
stream_loader = tn.Loader(tn.Batcher(SeriesStreamNode(seed=77), batch_size=4))

for step, indices_batch in enumerate(stream_loader):
    if step == 2:
        checkpoint = stream_loader.state_dict()
    if step == 3:
        after_ckpt = torch.stack(indices_batch)
        break

stream_loader.load_state_dict(checkpoint)
resumed = torch.stack(next(iter(stream_loader)))
print("infinite stream resumed mid-flight:", torch.equal(after_ckpt, resumed))

infinite stream resumed mid-flight: True


In [8]:
# It trains, of course — same loop, cap the steps because the stream never ends
node = SeriesStreamNode(seed=77)
node = tn.Batcher(node, batch_size=32)
node = tn.Mapper(node, map_fn=torch.stack)
model2 = NanoTST()
losses2 = train_with_loader(model2, tn.Loader(node), epochs=2, steps_per_epoch=14)

  Epoch 1 | loss: 1.6465 (14 steps)


  Epoch 2 | loss: 1.3865 (14 steps)


## Honest status & when to care (torchdata 0.11, July 2026)

- **Map-style migration: well paved.** The `nodes_dataloader` above is the docs' own recommended pattern.
- **`IterableDataset` support: still "coming soon"** — `get_worker_info` doesn't work inside `IterableWrapper`. The recommended pattern is what we did: iterate in the main process, parallelize the map (or write a small `BaseNode`).
- **Persistent workers: coming soon.**
- **Multi-dataset mixing exists today** (`MultiNodeWeightedSampler`) — the thing samplers never handled well.

**When to care now:** mid-epoch resume on long epochs, memory-bound multiprocessing, multi-dataset mixing. Otherwise the DataLoader is still fine — but now you know what's inside the box, because we just built it from parts.

Your Dataset survives all of this: nodes replace the *machinery*, not your data contract.